In [1]:
!pip install ultralytics -q

import ultralytics
ultralytics.checks()
print('\n✅ Setup complete!')

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 7035.7/8062.4 GB disk)

✅ Setup complete!


In [2]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/gbsaiganesh
/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1
/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1/valid
/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1/test
/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1/train


In [3]:
!find /kaggle/input -maxdepth 3

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/gbsaiganesh
/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1


In [4]:
import glob

print(glob.glob("/kaggle/input/**/_annotations.coco.json", recursive=True))

['/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1/valid/_annotations.coco.json', '/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1/test/_annotations.coco.json', '/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1/train/_annotations.coco.json']


In [5]:
import json
import shutil
from pathlib import Path


DATASET_DIR = Path("/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1")

# Output folder
OUTPUT_DIR = Path("yolo_dataset")


def coco_to_yolo(split):
    img_dir = DATASET_DIR / split
    ann_file = img_dir / "_annotations.coco.json"

    if not ann_file.exists():
        print(f"❌ {ann_file} not found")
        return 0

    out_img = OUTPUT_DIR / split / "images"
    out_lbl = OUTPUT_DIR / split / "labels"

    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)

    with open(ann_file, "r") as f:
        coco = json.load(f)

    # image_id -> image information
    img_map = {img["id"]: img for img in coco["images"]}

    # image_id -> annotations
    ann_map = {}
    for ann in coco["annotations"]:
        ann_map.setdefault(ann["image_id"], []).append(ann)

    # Category mapping
    categories = sorted(
        [c for c in coco["categories"] if c["id"] != 0],
        key=lambda x: x["id"]
    )

    cat_map = {c["id"]: i for i, c in enumerate(categories)}

    print(f"\n{split.upper()} CLASSES:")
    print([c["name"] for c in categories])

    converted = 0
    skipped = 0

    for img_id, img in img_map.items():

        image_name = img["file_name"]
        W = img["width"]
        H = img["height"]

        src = img_dir / image_name
        dst = out_img / image_name

        if not src.exists():
            skipped += 1
            continue

        shutil.copy2(src, dst)

        label_file = out_lbl / (Path(image_name).stem + ".txt")

        with open(label_file, "w") as lf:

            for ann in ann_map.get(img_id, []):

                if ann["category_id"] not in cat_map:
                    continue

                cls = cat_map[ann["category_id"]]

                x, y, w, h = ann["bbox"]

                xc = (x + w / 2) / W
                yc = (y + h / 2) / H
                wn = w / W
                hn = h / H

                lf.write(
                    f"{cls} "
                    f"{xc:.6f} "
                    f"{yc:.6f} "
                    f"{wn:.6f} "
                    f"{hn:.6f}\n"
                )

        converted += 1

    print(f"✅ {split}: {converted} images converted")

    if skipped:
        print(f"⚠️  {skipped} images missing")

    return converted


print("🔄 Converting COCO → YOLO")

for split in ["train", "valid", "test"]:
    coco_to_yolo(split)

print("\n🎉 Conversion Complete!")

print("\nDataset saved as:")

for split in ["train", "valid", "test"]:
    print(OUTPUT_DIR / split)

🔄 Converting COCO → YOLO

TRAIN CLASSES:
['Aphids', 'Bacterial Rot', 'Bacterial Spot', 'Blossom end rot', 'Early Blight', 'Fruit Crack', 'Fruit Damage', 'Leafhoppers and Jassids', 'Pest Damage', 'Potassium Deficiency', 'Viral', 'bactarial canker', 'bug', 'caterpilla', 'late blight', 'leafminer', 'powedry_mildew', 'spodoptera frugiperda', 'tuta', 'whiteflies']
✅ train: 4353 images converted

VALID CLASSES:
['Aphids', 'Bacterial Rot', 'Bacterial Spot', 'Blossom end rot', 'Early Blight', 'Fruit Crack', 'Fruit Damage', 'Leafhoppers and Jassids', 'Pest Damage', 'Potassium Deficiency', 'Viral', 'bactarial canker', 'bug', 'caterpilla', 'late blight', 'leafminer', 'powedry_mildew', 'spodoptera frugiperda', 'tuta', 'whiteflies']
✅ valid: 636 images converted

TEST CLASSES:
['Aphids', 'Bacterial Rot', 'Bacterial Spot', 'Blossom end rot', 'Early Blight', 'Fruit Crack', 'Fruit Damage', 'Leafhoppers and Jassids', 'Pest Damage', 'Potassium Deficiency', 'Viral', 'bactarial canker', 'bug', 'caterpilla

In [6]:
import os

for root, dirs, files in os.walk("yolo_dataset"):
    print(root)
    for f in files[:5]:
        print("   ", f)

yolo_dataset
yolo_dataset/valid
yolo_dataset/valid/labels
    Powdery-Mildew-on-Delphinium-Plants-HG_jpg.rf.28c648fabc91845df522df74fa60b7dd.txt
    r-2181-_jpg.rf.7b3719c282a1e8b0a6609119d159eae4.txt
    tomatogianttree1_1400x_jpg.rf.fdcbb39cfbc6b5019dd7baa0fafae0ec.txt
    r-2535-_jpg.rf.27b522a4d35497f8842b43b102a8552c.txt
    IMG_6476_jpg.rf.8c5fcfdfda20fdd3692c002bc21ff0bb.txt
yolo_dataset/valid/images
    20240303_121534_010_saved_jpg.rf.68fe62c2cd36e3be476654c044b7c669.jpg
    151Tomato_late_blight_jpg.rf.b9c1110ab3bf74b9fe192a9569c58052.jpg
    IMG20240305122350-1-_jpg.rf.c19dae5dcae6ce6e10101b86d44f4934.jpg
    IMG20240305123249-1-_jpg.rf.74989f50768dd84c164234c609599a16.jpg
    HA-16-_jpg.rf.d08a66ef411b8a195376ac094ab46a46.jpg
yolo_dataset/test
yolo_dataset/test/labels
    IMG_20190219_010159_6_jpg.rf.ec87b976e415758f2d82b0ba78bc90b6.txt
    119Bacterial_Spots_jpg.rf.754e00598321171407684d285c0d77be.txt
    potatoaphid-1-_jpg.rf.471e31393c013cdcfeedb21fb37c2137.txt
    HA-47

In [7]:
from pathlib import Path

for split in ["train", "valid", "test"]:
    img_count = len(list(Path(f"yolo_dataset/{split}/images").glob("*")))
    lbl_count = len(list(Path(f"yolo_dataset/{split}/labels").glob("*.txt")))

    print(f"{split}")
    print(f"Images : {img_count}")
    print(f"Labels : {lbl_count}\n")

train
Images : 4353
Labels : 4353

valid
Images : 636
Labels : 636

test
Images : 181
Labels : 181



In [8]:
import json

with open("/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1/train/_annotations.coco.json") as f:
    coco = json.load(f)

print("Number of classes:", len(coco["categories"]))

for c in coco["categories"]:
    print(c["id"], c["name"])

Number of classes: 21
0 Phase2-Phase3-tomato-Spodop-iEGw
1 Aphids
2 Bacterial Rot
3 Bacterial Spot
4 Blossom end rot
5 Early Blight
6 Fruit Crack
7 Fruit Damage
8 Leafhoppers and Jassids
9 Pest Damage
10 Potassium Deficiency
11 Viral
12 bactarial canker
13 bug
14 caterpilla
15 late blight
16 leafminer
17 powedry_mildew
18 spodoptera frugiperda
19 tuta
20 whiteflies


In [9]:
import json
from pathlib import Path

# ---- YOUR DATASET PATH ----
dataset_path = "/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1"

# Read COCO annotation
ann_file = f"{dataset_path}/train/_annotations.coco.json"

with open(ann_file, "r") as f:
    coco = json.load(f)

# Extract categories
categories = sorted(coco["categories"], key=lambda x: x["id"])

# Build YAML text
yaml_text = """path: /kaggle/working/yolo_dataset
train: train/images
val: valid/images
test: test/images

names:
"""

for i, cat in enumerate(categories):
    yaml_text += f"  {i}: {cat['name']}\n"

# SAVE THE FILE
yaml_path = "/kaggle/working/data.yaml"
Path(yaml_path).write_text(yaml_text)

print("✅ data.yaml saved at:", yaml_path)
print("\n----- YAML CONTENT -----\n")
print(yaml_text)

✅ data.yaml saved at: /kaggle/working/data.yaml

----- YAML CONTENT -----

path: /kaggle/working/yolo_dataset
train: train/images
val: valid/images
test: test/images

names:
  0: Phase2-Phase3-tomato-Spodop-iEGw
  1: Aphids
  2: Bacterial Rot
  3: Bacterial Spot
  4: Blossom end rot
  5: Early Blight
  6: Fruit Crack
  7: Fruit Damage
  8: Leafhoppers and Jassids
  9: Pest Damage
  10: Potassium Deficiency
  11: Viral
  12: bactarial canker
  13: bug
  14: caterpilla
  15: late blight
  16: leafminer
  17: powedry_mildew
  18: spodoptera frugiperda
  19: tuta
  20: whiteflies



In [10]:
import torch
from ultralytics import YOLO


print("CUDA available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

CUDA available: True
Number of GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [11]:
model = YOLO('yolov8n.pt')  

print('🚀 Starting multi-GPU training on 2×T4...')

results = model.train(
    data     = '/kaggle/working/data.yaml', 
    epochs   = 50,
    imgsz    = 640,
    batch    = 32,        
    name     = 'pest_detector',
    project  = '/kaggle/working/runs/detect',
    patience = 15,

   
    device   = [0, 1],

    augment  = True,
    workers  = 8,
    verbose  = True,
    exist_ok = True
)

print('\n✅ Training complete!')
print('Best model:')
print('/kaggle/working/runs/detect/pest_detector/weights/best.pt')

🚀 Starting multi-GPU training on 2×T4...
Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.9

In [12]:
metrics = model.val()

print("========== FINAL RESULTS ==========")
print("mAP@50      :", round(metrics.box.map50, 4))
print("mAP@50-95   :", round(metrics.box.map, 4))
print("Precision   :", round(metrics.box.mp, 4))
print("Recall      :", round(metrics.box.mr, 4))

Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 3,009,743 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2062.4±1275.4 MB/s, size: 799.7 KB)
val: Scanning /kaggle/working/yolo_dataset/valid/labels.cache... 636 images, 169 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 636/636 222.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 40/40 2.5it/s 16.1s0.3s
                   all        636       1195      0.634       0.34      0.365      0.192
Phase2-Phase3-tomato-Spodop-iEGw          4         20          1          0      0.046     0.0364
        Bacterial Spot          2          2          1          0      0.099     0.0297
          Fruit Damage          4          4          1          0       0.24      0.111
  Potassium Deficiency          1          1          0          0          0          0
                

In [17]:
model = YOLO('yolov8s.pt')  

print('🚀 Starting multi-GPU training on 2×T4...')

results = model.train(
    data     = '/kaggle/working/data.yaml', 
    epochs   = 50,
    imgsz    = 640,
    batch    = 32,        
    name     = 'pest_detector',
    project  = '/kaggle/working/runs/detect',
    patience = 15,

   
    device   = [0, 1],

    augment  = True,
    workers  = 8,
    verbose  = True,
    exist_ok = True
)

print('\n✅ Training complete!')
print('Best model:')
print('/kaggle/working/runs/detect/pest_detector/weights/best.pt')

🚀 Starting multi-GPU training on 2×T4...
Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.9

In [18]:
metrics = model.val()

print("========== FINAL RESULTS ==========")
print("mAP@50      :", round(metrics.box.map50, 4))
print("mAP@50-95   :", round(metrics.box.map, 4))
print("Precision   :", round(metrics.box.mp, 4))
print("Recall      :", round(metrics.box.mr, 4))

Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 11,133,711 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2539.5±1568.3 MB/s, size: 637.6 KB)
val: Scanning /kaggle/working/yolo_dataset/valid/labels.cache... 636 images, 169 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 636/636 242.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 40/40 3.0it/s 13.3s0.3s
                   all        636       1195      0.414      0.485      0.465      0.228
Phase2-Phase3-tomato-Spodop-iEGw          4         20      0.285       0.05     0.0723     0.0378
        Bacterial Spot          2          2      0.356      0.356      0.663      0.148
          Fruit Damage          4          4      0.377        0.5      0.422      0.251
  Potassium Deficiency          1          1          0          0          0          0
              

In [11]:
from ultralytics import RTDETR

# Load pretrained RT-DETR Large model
model = RTDETR('rtdetr-l.pt')

print('🚀 Starting RT-DETR multi-GPU training on 2×T4...')

results = model.train(

    # Dataset
    data='/kaggle/working/data.yaml',

    # Training
    epochs=50,
    imgsz=640,
    batch=16,              # RT-DETR uses more GPU memory

    # Save
    name='rtdetr_pest_detector',
    project='/kaggle/working/runs/detect',
    exist_ok=True,

    # Hardware
    device=[0, 1],         # Use both T4 GPUs
    workers=8,

    # Optimization
    optimizer='AdamW',     # Recommended for transformer models
    lr0=1e-4,
    patience=15,

    # Augmentation
    augment=True,
    mosaic=1.0,
    mixup=0.2,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.1,
    scale=0.5,
    shear=2,

    # Logging
    verbose=True,
    plots=True,
    save=True
)

print('\n✅ RT-DETR training complete!')
print('Best model:')
print('/kaggle/working/runs/detect/rtdetr_pest_detector/weights/best.pt')

🚀 Starting RT-DETR multi-GPU training on 2×T4...
Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=10, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mo

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

       1/50      6.93G      1.436       1.38      1.053          0        640: 100% ━━━━━━━━━━━━ 273/273 1.1it/s 3:591.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 1.4it/s 14.0s0.5s
                   all        636       1195   3.76e-05     0.0212   2.26e-05   3.28e-06

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

       2/50      7.13G      1.292     0.7362     0.9067          5        640: 100% ━━━━━━━━━━━━ 273/273 1.2it/s 3:390.7ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 1.7it/s 12.0s.4s
                   all        636       1195   9.26e-05     0.0523   0.000381   0.000101

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


...

      48/50      7.05G     0.5803     0.6458     0.3954          1        640: 100% ━━━━━━━━━━━━ 273/273 1.4it/s 3:100.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 1.8it/s 11.0s.5s
                   all        636       1195      0.297      0.301      0.231      0.103

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

      49/50      7.05G     0.5924     0.6437     0.3822          0        640: 100% ━━━━━━━━━━━━ 273/273 1.4it/s 3:140.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 1.8it/s 10.8s.5s
                   all        636       1195      0.309      0.296      0.231      0.105

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

      50/50      7.05G     0.6048     0.6362      0.389          3        640: 100% ━━━━━━━━━━━━ 273/273 1.4it/s 3:120.7ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 1.8it/s 10.9s.5s
                   all        636       1195      0.394      0.289      0.221        0.1

50 epochs completed in 3.132 hours.
Optimizer stripped from /kaggle/working/runs/detect/rtdetr_pest_detector/weights/last.pt, 66.3MB
Optimizer stripped from /kaggle/working/runs/detect/rtdetr_pest_detector/weights/best.pt, 66.3MB

Validating /kaggle/working/runs/detect/rtdetr_pest_detector/weights/best.pt...
rt-detr-l summary: 310 layers, 32,026,895 parameters, 0 gradients, 103.5 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 1.5it/s 13.2s0.5s
                   all        636       1195        0.5      0.264      0.234       0.11
Phase2-Phase3-tomato-Spodop-iEGw          4  

In [12]:
from ultralytics import RTDETR

# Load the BEST trained RT-DETR model
best_model = RTDETR(
    '/kaggle/working/runs/detect/rtdetr_pest_detector/weights/best.pt'
)

print("✅ Best RT-DETR model loaded!")

# Run validation
metrics = best_model.val(
    data='/kaggle/working/data.yaml'
)

print("\n" + "="*30)
print("🐛 RT-DETR FINAL RESULTS")
print("="*30)

print(f"mAP@50      : {metrics.box.map50:.4f}")
print(f"mAP@50-95   : {metrics.box.map:.4f}")
print(f"Precision   : {metrics.box.mp:.4f}")
print(f"Recall      : {metrics.box.mr:.4f}")

✅ Best RT-DETR model loaded!
Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rt-detr-l summary: 310 layers, 32,026,895 parameters, 0 gradients, 103.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2895.5±1673.0 MB/s, size: 799.7 KB)
val: Scanning /kaggle/working/yolo_dataset/valid/labels.cache... 636 images, 169 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 636/636 205.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 40/40 1.5it/s 26.7s0.6ss
                   all        636       1195      0.501      0.269      0.235      0.111
Phase2-Phase3-tomato-Spodop-iEGw          4         20          1          0    0.00192   0.000908
        Bacterial Spot          2          2          1          0     0.0663    0.00993
          Fruit Damage          4          4          1          0     0.0827     0.0298
  Potassium Deficiency          1          1          0          0         

In [13]:
metrics = model.val()

print("========== FINAL RESULTS ==========")
print("mAP@50      :", round(metrics.box.map50, 4))
print("mAP@50-95   :", round(metrics.box.map, 4))
print("Precision   :", round(metrics.box.mp, 4))
print("Recall      :", round(metrics.box.mr, 4))

Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rt-detr-l summary: 310 layers, 32,026,895 parameters, 0 gradients, 103.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3551.5±1225.5 MB/s, size: 934.3 KB)
val: Scanning /kaggle/working/yolo_dataset/valid/labels.cache... 636 images, 169 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 636/636 190.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 40/40 1.5it/s 26.5s0.6ss
                   all        636       1195      0.501      0.269      0.235      0.111
Phase2-Phase3-tomato-Spodop-iEGw          4         20          1          0    0.00192   0.000908
        Bacterial Spot          2          2          1          0     0.0663    0.00993
          Fruit Damage          4          4          1          0     0.0827     0.0298
  Potassium Deficiency          1          1          0          0          0          0
               

In [ ]:
from ultralytics import YOLO

# Load YOLO11 Small model
model = YOLO('yolo11s.pt')

print('🚀 Starting YOLO11s multi-GPU training on 2×T4...')

results = model.train(

    # Dataset
    data='/kaggle/working/data.yaml',

    # Training
    epochs=50,
    imgsz=640,
    batch=32,

    # Save
    name='yolo11s_pest_detector',
    project='/kaggle/working/runs/detect',
    exist_ok=True,

    # Hardware
    device=[0, 1],
    workers=8,

    # Optimization
    optimizer='auto',
    patience=15,

    # Augmentation
    augment=True,
    mosaic=1.0,
    mixup=0.2,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.1,
    scale=0.5,
    shear=2,

    # Logging
    verbose=True,
    plots=True,
    save=True
)

print('\n✅ YOLO11s training complete!')
print('Best model:')
print('/kaggle/working/runs/detect/yolo11s_pest_detector/weights/best.pt')

🚀 Starting YOLO11s multi-GPU training on 2×T4...
Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=10, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det

In [13]:
# Validate the best YOLO11 model
metrics = model.val(
    data='/kaggle/working/data.yaml'
)

print('\n' + '='*50)
print('YOLO11s FINAL RESULTS')
print('='*50)

print(f"mAP@50      : {metrics.box.map50:.4f}")
print(f"mAP@50-95   : {metrics.box.map:.4f}")
print(f"Precision   : {metrics.box.mp:.4f}")
print(f"Recall      : {metrics.box.mr:.4f}")

Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11m summary (fused): 126 layers, 20,046,223 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2616.8±1550.4 MB/s, size: 799.7 KB)
val: Scanning /kaggle/working/yolo_dataset/valid/labels.cache... 636 images, 169 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 636/636 156.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 40/40 1.7it/s 23.9s0.6ss
                   all        636       1195      0.578      0.408      0.415      0.203
Phase2-Phase3-tomato-Spodop-iEGw          4         20          0          0     0.0341    0.00875
        Bacterial Spot          2          2          1          0          0          0
          Fruit Damage          4          4          1      0.454      0.648      0.192
  Potassium Deficiency          1          1          0          0          0          0
          

In [14]:
import os

for split in ["train", "valid", "test"]:

    img_path = f"/kaggle/working/yolo_dataset{split}/images"
    lbl_path = f"/kaggle/working/yolo_dataset/{split}/labels"

    print("\n", split.upper())
    print("Images :", len(os.listdir(img_path)) if os.path.exists(img_path) else 0)
    print("Labels :", len(os.listdir(lbl_path)) if os.path.exists(lbl_path) else 0)


 TRAIN
Images : 0
Labels : 4353

 VALID
Images : 0
Labels : 636

 TEST
Images : 0
Labels : 181
